# 03 - Carga de datos en Snowflake
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook carga el CSV limpio generado en el paso anterior en Snowflake usando un **esquema en estrella**.

**Entrada:** `data/processed/contratos_limpios.csv`  
**Destino:** 4 tablas de dimensiones + `FACT_CONTRATOS` en Snowflake

Las credenciales se leen del fichero `.env` — nunca se escriben directamente en el código.

## 0 — Instalar dependencias
Ejecuta solo la primera vez.

In [35]:
%pip install --prefer-binary -q snowflake-connector-python python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## 1 — Importaciones y carga de credenciales

In [36]:
import os
import pandas as pd
import snowflake.connector
from pathlib import Path
from dotenv import load_dotenv

# Cargamos las variables del fichero .env en el entorno del proceso
load_dotenv(Path("../.env"))

# Leemos cada credencial del entorno — si alguna está vacía, lo avisamos
ACCOUNT   = os.getenv("SNOWFLAKE_ACCOUNT")
USER      = os.getenv("SNOWFLAKE_USER")
PASSWORD  = os.getenv("SNOWFLAKE_PASSWORD")
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
DATABASE  = os.getenv("SNOWFLAKE_DATABASE")
SCHEMA    = os.getenv("SNOWFLAKE_SCHEMA")

# Verificamos que todas las variables están rellenas antes de continuar
vacias = [k for k, v in {
    "ACCOUNT": ACCOUNT, "USER": USER, "PASSWORD": PASSWORD,
    "WAREHOUSE": WAREHOUSE, "DATABASE": DATABASE, "SCHEMA": SCHEMA
}.items() if not v]

if vacias:
    raise ValueError(f"Faltan credenciales en .env: {vacias}")

print(f"Cuenta:    {ACCOUNT}")
print(f"Usuario:   {USER}")
print(f"Warehouse: {WAREHOUSE}")
print(f"Base:      {DATABASE}.{SCHEMA}")
print("Credenciales cargadas correctamente.")

Cuenta:    aisfeil-lu45385
Usuario:   socratesagudo
Warehouse: FRAUDE_VILAGARCIA
Base:      FRAUDE_VILAGARCIA.PUBLIC
Credenciales cargadas correctamente.


## 2 — Conectar a Snowflake

In [37]:
# Abrimos la conexión con Snowflake usando las credenciales del .env
conn = snowflake.connector.connect(
    account   = ACCOUNT,
    user      = USER,
    password  = PASSWORD,
    warehouse = WAREHOUSE,
    database  = DATABASE,
    schema    = SCHEMA,
)

# El cursor es el objeto que ejecuta las sentencias SQL
cur = conn.cursor()

print("Conexión establecida.")
cur.execute("SELECT CURRENT_VERSION()")
version = cur.fetchone()[0]
print(f"Versión de Snowflake: {version}")

Conexión establecida.
Versión de Snowflake: 10.13.104


## 3 — Crear el esquema en estrella

Se crean 4 tablas de dimensiones y 1 tabla de hechos para facilitar el análisis en Power BI.

| Tabla | Tipo | Descripción |
|---|---|---|
| `DIM_CONTRATISTA` | Dimensión | Empresas y autónomos que reciben contratos |
| `DIM_FECHA` | Dimensión | Calendario con atributos temporales |
| `DIM_TIPO_CONTRATO` | Dimensión | Clasificación del contrato |
| `DIM_ENTIDAD` | Dimensión | Organismo público contratante |
| `FACT_CONTRATOS` | Hechos | Métricas económicas por contrato |

In [38]:
# Creamos las 4 dimensiones y la tabla de hechos
# CREATE OR REPLACE garantiza que el esquema esté siempre actualizado al re-ejecutar

DDL_DIM_CONTRATISTA = """
CREATE OR REPLACE TABLE DIM_CONTRATISTA (
    nif_contratista    VARCHAR PRIMARY KEY,  -- NIF o CIF del contratista
    nombre_contratista VARCHAR,              -- razón social o nombre
    nif_valido         BOOLEAN,             -- True si pasa la validación de checksum
    motivo_invalido    VARCHAR,             -- descripción del error si nif_valido = False
    contratistas_raw   VARCHAR              -- campo original para trazabilidad
)
"""

DDL_DIM_FECHA = """
CREATE OR REPLACE TABLE DIM_FECHA (
    fecha       DATE PRIMARY KEY,  -- clave natural — la fecha completa
    anio        INTEGER,
    trimestre   INTEGER,           -- 1 a 4
    mes         INTEGER,           -- 1 a 12
    nombre_mes  VARCHAR,           -- 'Enero', 'Febrero'...
    dia_mes     INTEGER,           -- 1 a 31
    dia_semana  VARCHAR            -- 'Lunes', 'Martes'...
)
"""

DDL_DIM_TIPO_CONTRATO = """
CREATE OR REPLACE TABLE DIM_TIPO_CONTRATO (
    tipo_contrato_key    INTEGER PRIMARY KEY,  -- clave sustituta
    tipo_contrato        VARCHAR,              -- Obras / Servicios / Suministros
    tipo_procedimiento   VARCHAR,
    sistema_contratacion VARCHAR,
    tramitacion          VARCHAR               -- Ordinaria / Urgente / Emergencia
)
"""

DDL_DIM_ENTIDAD = """
CREATE OR REPLACE TABLE DIM_ENTIDAD (
    entidad_contratante VARCHAR PRIMARY KEY  -- nombre del organismo público
)
"""

DDL_FACT = """
CREATE OR REPLACE TABLE FACT_CONTRATOS (
    num_referencia       VARCHAR PRIMARY KEY,  -- identificador único del expediente
    anio_contrato        INTEGER,
    num_contrato_anio    INTEGER,              -- secuencial dentro del año
    fecha_formalizacion  DATE,                -- FK → DIM_FECHA (puede ser nula)
    fecha_estimada       DATE,                -- FK → DIM_FECHA — siempre rellena
    entidad_contratante  VARCHAR,             -- FK → DIM_ENTIDAD
    tipo_contrato_key    INTEGER,             -- FK → DIM_TIPO_CONTRATO
    nif_contratista      VARCHAR,             -- FK → DIM_CONTRATISTA
    objeto_contrato      VARCHAR,
    codigo_cpv           VARCHAR,
    plazo_meses          FLOAT,
    valor_estimado_eur   FLOAT,
    presupuesto_base_eur FLOAT,
    importe_sin_iva_eur  FLOAT,
    importe_con_iva_eur  FLOAT,
    flag_limite          VARCHAR,             -- ok / cerca_del_limite / supera_limite
    observaciones        VARCHAR,
    url_licitacion       VARCHAR
)
"""

for nombre, ddl in [
    ("DIM_CONTRATISTA",   DDL_DIM_CONTRATISTA),
    ("DIM_FECHA",         DDL_DIM_FECHA),
    ("DIM_TIPO_CONTRATO", DDL_DIM_TIPO_CONTRATO),
    ("DIM_ENTIDAD",       DDL_DIM_ENTIDAD),
    ("FACT_CONTRATOS",    DDL_FACT),
]:
    cur.execute(ddl)
    print(f"Tabla {nombre} creada correctamente.")

Tabla DIM_CONTRATISTA creada correctamente.
Tabla DIM_FECHA creada correctamente.
Tabla DIM_TIPO_CONTRATO creada correctamente.
Tabla DIM_ENTIDAD creada correctamente.
Tabla FACT_CONTRATOS creada correctamente.


## 4 — Cargar el CSV y construir las dimensiones

In [39]:
RUTA_CSV = Path("../data/processed/contratos_limpios.csv")

# Leemos con parse_dates para poder calcular atributos del calendario en DIM_FECHA
df = pd.read_csv(
    RUTA_CSV,
    encoding    = "utf-8-sig",
    parse_dates = ["fecha_formalizacion", "fecha_estimada"],
    dtype       = {"anio_contrato": "Int64", "num_contrato_anio": "Int64"},
)

print(f"Filas cargadas: {len(df)}")
df.head(3)

Filas cargadas: 607


,num_referencia,anio_contrato,num_contrato_anio,fecha_formalizacion,fecha_estimada,entidad_contratante,tipo_contrato,tipo_procedimiento,sistema_contratacion,tramitacion,...,importe_sin_iva_eur,importe_con_iva_eur,flag_limite,nif_contratista,nif_valido,motivo_invalido,nombre_contratista,contratistas_raw,observaciones,url_licitacion
0,CT 02/2023,2023,1,NaT,2023-02-02,Ayuntamiento Vilagarcía de Arousa,Suministros,Contrato menor,No aplica,Ordinaria,...,2394.00,2896.74,ok,B36127405,True,NaN,REDINOR S.L.,B36127405 - REDINOR S.L.,NaN,https://contrataciondelestado.es/wps/poc?uri=d...
1,CT 75/2022,2022,2,NaT,2022-10-10,Ayuntamiento Vilagarcía de Arousa,Servicios,Contrato menor,No aplica,Ordinaria,...,14849.00,17967.29,cerca_del_limite,35458826W,True,NaN,MANUEL TANOIRA CARBALLO,35458826W - MANUEL TANOIRA CARBALLO,NaN,https://contrataciondelestado.es/wps/poc?uri=d...
2,CT 05/2023,2023,2,NaT,2023-08-27,Ayuntamiento Vilagarcía de Arousa,Suministros,Contrato menor,No aplica,Ordinaria,...,5559.96,6727.56,ok,B15090061,True,NaN,"TARRIO Y SUAREZ, SL","B15090061 - TARRIO Y SUAREZ, SL",NaN,https://contrataciondelestado.es/wps/poc?uri=d...


In [40]:
# --- DIM_CONTRATISTA ---
# Un registro por NIF único; descartamos filas sin NIF
dim_contratista = (
    df[["nif_contratista", "nombre_contratista", "nif_valido", "motivo_invalido", "contratistas_raw"]]
    .dropna(subset=["nif_contratista"])
    .drop_duplicates(subset=["nif_contratista"])
    .reset_index(drop=True)
)

# --- DIM_FECHA ---
# Recogemos todas las fechas únicas de ambas columnas antes de convertirlas a date
todas_fechas = (
    pd.concat([df["fecha_formalizacion"].dropna(), df["fecha_estimada"].dropna()])
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}
DIAS_ES = {
    0: "Lunes", 1: "Martes", 2: "Miercoles", 3: "Jueves",
    4: "Viernes", 5: "Sabado", 6: "Domingo",
}

dim_fecha = pd.DataFrame({
    "fecha":      todas_fechas.dt.date,         # DATE nativo de Python
    "anio":       todas_fechas.dt.year,
    "trimestre":  todas_fechas.dt.quarter,
    "mes":        todas_fechas.dt.month,
    "nombre_mes": todas_fechas.dt.month.map(MESES_ES),
    "dia_mes":    todas_fechas.dt.day,
    "dia_semana": todas_fechas.dt.dayofweek.map(DIAS_ES),
})

# --- DIM_TIPO_CONTRATO ---
# Combinación única de los 4 atributos de clasificación del contrato
dim_tipo = (
    df[["tipo_contrato", "tipo_procedimiento", "sistema_contratacion", "tramitacion"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
# Clave sustituta: entero secuencial (1, 2, 3, ...)
dim_tipo.insert(0, "tipo_contrato_key", range(1, len(dim_tipo) + 1))

# --- DIM_ENTIDAD ---
dim_entidad = (
    df[["entidad_contratante"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"DIM_CONTRATISTA:   {len(dim_contratista)} filas")
print(f"DIM_FECHA:         {len(dim_fecha)} filas")
print(f"DIM_TIPO_CONTRATO: {len(dim_tipo)} filas")
print(f"DIM_ENTIDAD:       {len(dim_entidad)} filas")

DIM_CONTRATISTA:   334 filas
DIM_FECHA:         340 filas
DIM_TIPO_CONTRATO: 4 filas
DIM_ENTIDAD:       1 filas


## 5 — Insertar dimensiones en Snowflake

In [41]:
import numpy as np

def normalizar(valor):
    """Convierte tipos pandas/numpy a tipos Python nativos que acepta el conector de Snowflake."""
    if valor is None:
        return None
    try:
        if pd.isna(valor):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(valor, pd.Timestamp):
        return valor.date()
    if isinstance(valor, np.integer):
        return int(valor)
    if isinstance(valor, np.floating):
        return float(valor)
    if isinstance(valor, np.bool_):
        return bool(valor)
    return valor

def insertar_tabla(nombre_tabla, df_dim, batch_size=1000):
    """Inserta un DataFrame completo en Snowflake en lotes para no superar el límite de mensaje."""
    columnas  = ", ".join(df_dim.columns)
    wildcards = ", ".join(["%s"] * len(df_dim.columns))
    sql = f"INSERT INTO {nombre_tabla} ({columnas}) VALUES ({wildcards})"

    filas = [
        tuple(normalizar(v) for v in row)
        for row in df_dim.itertuples(index=False, name=None)
    ]

    for i in range(0, len(filas), batch_size):
        cur.executemany(sql, filas[i : i + batch_size])

    conn.commit()
    print(f"  {nombre_tabla:<22} {len(filas)} filas insertadas.")

print("Insertando dimensiones:")
insertar_tabla("DIM_CONTRATISTA",   dim_contratista)
insertar_tabla("DIM_FECHA",         dim_fecha)
insertar_tabla("DIM_TIPO_CONTRATO", dim_tipo)
insertar_tabla("DIM_ENTIDAD",       dim_entidad)
print("Dimensiones cargadas correctamente.")

Insertando dimensiones:
  DIM_CONTRATISTA        334 filas insertadas.
  DIM_FECHA              340 filas insertadas.
  DIM_TIPO_CONTRATO      4 filas insertadas.
  DIM_ENTIDAD            1 filas insertadas.
Dimensiones cargadas correctamente.


## 6 — Construir e insertar la tabla de hechos

In [42]:
# Añadimos la clave sustituta de DIM_TIPO_CONTRATO al DataFrame principal
df_fact = df.merge(
    dim_tipo[["tipo_contrato_key", "tipo_contrato", "tipo_procedimiento",
              "sistema_contratacion", "tramitacion"]],
    on  = ["tipo_contrato", "tipo_procedimiento", "sistema_contratacion", "tramitacion"],
    how = "left",
)

# Convertimos las fechas a datetime.date nativo — el conector no acepta pd.Timestamp
for col in ["fecha_formalizacion", "fecha_estimada"]:
    df_fact[col] = df_fact[col].apply(lambda x: x.date() if pd.notna(x) else None)

# Convertimos enteros nullable de pandas a int nativo o None
for col in df_fact.select_dtypes(include=["Int64", "Int32"]).columns:
    df_fact[col] = df_fact[col].apply(lambda x: None if pd.isna(x) else int(x))

# Reemplazamos cualquier NA restante por None para que Snowflake lo interprete como NULL
df_fact = df_fact.where(pd.notna(df_fact), other=None)

# Seleccionamos solo las columnas de la tabla de hechos en el orden correcto
COLS_FACT = [
    "num_referencia", "anio_contrato", "num_contrato_anio",
    "fecha_formalizacion", "fecha_estimada",
    "entidad_contratante", "tipo_contrato_key", "nif_contratista",
    "objeto_contrato", "codigo_cpv", "plazo_meses",
    "valor_estimado_eur", "presupuesto_base_eur", "importe_sin_iva_eur", "importe_con_iva_eur",
    "flag_limite", "observaciones", "url_licitacion",
]
df_fact = df_fact[COLS_FACT]

print(f"Insertando {len(df_fact)} filas en FACT_CONTRATOS...")
insertar_tabla("FACT_CONTRATOS", df_fact)

Insertando 607 filas en FACT_CONTRATOS...
  FACT_CONTRATOS         607 filas insertadas.


## 7 — Verificación de la carga

In [43]:
# Comprobamos que las filas de FACT_CONTRATOS coinciden con el CSV
cur.execute("SELECT COUNT(*) FROM FACT_CONTRATOS")
n_fact = cur.fetchone()[0]
print(f"Filas en CSV:            {len(df)}")
print(f"Filas en FACT_CONTRATOS: {n_fact}")
print(f"Coinciden: {'OK' if len(df) == n_fact else 'ERROR — revisar'}")
print()

# Mostramos el conteo de cada dimensión
for tabla in ["DIM_CONTRATISTA", "DIM_FECHA", "DIM_TIPO_CONTRATO", "DIM_ENTIDAD"]:
    cur.execute(f"SELECT COUNT(*) FROM {tabla}")
    n = cur.fetchone()[0]
    print(f"  {tabla:<22} {n} filas")

Filas en CSV:            607
Filas en FACT_CONTRATOS: 607
Coinciden: OK

  DIM_CONTRATISTA        334 filas
  DIM_FECHA              340 filas
  DIM_TIPO_CONTRATO      4 filas
  DIM_ENTIDAD            1 filas


In [44]:
# Vista previa con JOIN entre la fact y las dimensiones
# Así es exactamente como Power BI va a consumir los datos
cur.execute("""
    SELECT
        f.num_referencia,
        f.anio_contrato,
        t.tipo_contrato,
        t.tramitacion,
        f.presupuesto_base_eur,
        c.nombre_contratista,
        c.nif_valido,
        f.flag_limite
    FROM FACT_CONTRATOS f
    JOIN DIM_TIPO_CONTRATO t ON f.tipo_contrato_key = t.tipo_contrato_key
    JOIN DIM_CONTRATISTA   c ON f.nif_contratista   = c.nif_contratista
    LIMIT 5
""")
muestra = cur.fetchall()

cabecera = ["num_referencia", "anio", "tipo", "tramitacion", "presupuesto", "contratista", "nif_valido", "flag"]
print("Muestra con JOIN entre tablas:")
print("  " + "  |  ".join(f"{c:<15}" for c in cabecera))
print("  " + "-" * 110)
for fila in muestra:
    print("  " + "  |  ".join(f"{str(v):<15}" for v in fila))

Muestra con JOIN entre tablas:
  num_referencia   |  anio             |  tipo             |  tramitacion      |  presupuesto      |  contratista      |  nif_valido       |  flag           
  --------------------------------------------------------------------------------------------------------------
  CT 02/2023       |  2023             |  Suministros      |  Ordinaria        |  2394.0           |  REDINOR S.L.     |  True             |  ok             
  CT 75/2022       |  2022             |  Servicios        |  Ordinaria        |  14849.0          |  MANUEL TANOIRA CARBALLO  |  True             |  cerca_del_limite
  CT 05/2023       |  2023             |  Suministros      |  Ordinaria        |  5559.96          |  TARRIO Y SUAREZ, SL  |  True             |  ok             
  CT 66/2022-e     |  2022             |  Obras            |  Ordinaria        |  39990.2          |  Serviplustotal S.L.  |  True             |  cerca_del_limite
  CT 13/2023       |  2023             |  Servic

## 8 — Cerrar la conexión

In [45]:
# Cerramos cursor y conexión para liberar recursos
cur.close()
conn.close()
print("Conexión cerrada.")

Conexión cerrada.
